In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATAPATH = "/home/tim/precompute/use-cases/thread-sanitizer/sample_apps/performance-eval/results"

In [ ]:
df = pd.read_csv(DATAPATH + "/results_lulesh.csv")
# Extract size and iterations using regex
df['size'] = df['config'].str.extract(r'-s\s+(\d+)').astype(int)
df['iterations'] = df['config'].str.extract(r'-i\s+(\d+)').astype(int)
# time is in seconds: to float
df['time'] = df['time'].str.replace('s', '').astype(float)


In [ ]:
df

In [ ]:
iter_val = 20
df_filtered = df[df["iterations"] == iter_val]

for size_val, df_group in df_filtered.groupby("size"):
    # Create the line plot
    sns.lineplot(data=df_group, x="num_threads", y="time", hue="mode", marker="o")

    plt.title(f"LULESH: Execution Time (size = {size_val})")
    plt.xlabel("num Threads")
    plt.ylabel("Time (s)")
    plt.legend(title="Tsan build")
    plt.tight_layout()
    plt.savefig(f"lulesh_time_size_{size_val}.pdf")
    plt.show()
    plt.close()

In [ ]:
iter_val = 20
df_filtered = df[df["iterations"] == iter_val]
for size_val, df_group in df_filtered.groupby("size"):
    mean_times = df_group.groupby(["iterations", "num_threads", "mode"])["time"].mean().reset_index()
    # Pivot to wide format*: one row per config, columns per mode
    pivoted = mean_times.pivot(index="num_threads", columns="mode", values="time")
    # Normalize all modes relative to 'without'
    percentages = pivoted.div(pivoted["without"], axis=0)

    percentages[["modified", "normal"]].plot(kind="line", marker="o")

    plt.title(f"LULESH: Slowdown (size = {size_val})")
    plt.xlabel("problem Size")
    plt.ylabel("Slowdown Factor")
    plt.legend(title="Tsan build")
    plt.tight_layout()
    plt.savefig(f"lulesh_slowdown_size_{size_val}.pdf")
    plt.show()
    plt.close()

In [ ]:
mean_times = df_filtered.groupby(["iterations", "size", "mode"])["time"].mean().reset_index()
# Pivot to wide format*: one row per config, columns per mode
pivoted = mean_times.pivot(index="size", columns="mode", values="time")
# Normalize all modes relative to 'without'
percentages = pivoted.div(pivoted["without"], axis=0)

percentages[["modified", "normal"]].plot(kind="line", marker="o")

plt.title(f"LULESH: Slowdown (iterations = {iter_val})")
plt.xlabel("problem Size")
plt.ylabel("Slowdown Factor")
plt.legend(title="Tsan build")
plt.tight_layout()
plt.savefig("lulesh_slowdown.pdf")

In [ ]:
df_hpccg = pd.read_csv(DATAPATH + "/results_hpccg.csv")
# Extract size and iterations using regex
df_hpccg['size'] = df_hpccg['config'].str.extract(r'(\d+)').astype(int)
# time is in seconds: to float
df_hpccg['time'] = df_hpccg['time'].str.replace('s', '').astype(float)

In [ ]:
# Create the line plot
sns.lineplot(data=df_hpccg, x="size", y="time", hue="mode", marker="o")

plt.title(f"HPCCG: Execution Time")
plt.xlabel("Size")
plt.ylabel("Time (s)")
plt.legend(title="Tsan build")
plt.tight_layout()
plt.savefig("hpccg_time.pdf")

In [ ]:
for size_val, df_group in df_hpccg.groupby("size"):
    sns.lineplot(data=df_group, x="num_threads", y="time", hue="mode", marker="o")

    plt.title(f"HPCCG: Execution Time (size={size_val})")
    plt.xlabel("num threads")
    plt.ylabel("Time (s)")
    plt.legend(title="Tsan build")
    plt.tight_layout()
    plt.savefig(f"hpccg_time_size{size_val}.pdf")
    plt.show()
    plt.close()

In [ ]:
mean_times = df_hpccg.groupby(["size", "mode"])["time"].mean().reset_index()
# Pivot to wide format*: one row per config, columns per mode
pivoted = mean_times.pivot(index="size", columns="mode", values="time")
# Normalize all modes relative to 'without'
percentages = pivoted.div(pivoted["without"], axis=0)

percentages[["modified", "normal"]].plot(kind="line", marker="o")

plt.title(f"HPCCG: Slowdown")
plt.xlabel("problem Size")
plt.ylabel("Slowdown Factor")
plt.legend(title="Tsan build")
plt.tight_layout()
plt.savefig("hpccg_slowdown.pdf")

In [ ]:
df_filtered = df_hpccg[df_hpccg["size"] <= 200]  # larger sizes no measurement val
for size_val, df_group in df_filtered.groupby("size"):
    mean_times = df_group.groupby(["num_threads", "mode"])["time"].mean().reset_index()
    # Pivot to wide format*: one row per config, columns per mode
    pivoted = mean_times.pivot(index="num_threads", columns="mode", values="time")
    # Normalize all modes relative to 'without'
    percentages = pivoted.div(pivoted["without"], axis=0)

    percentages[["modified", "normal"]].plot(kind="line", marker="o")

    plt.title(f"HPCCG: Slowdown (size={size_val})")
    plt.xlabel("num threads")
    plt.ylabel("Slowdown Factor")
    plt.legend(title="Tsan build")
    plt.tight_layout()
    plt.savefig(f"hpccg_slowdown_size{size_val}.pdf")
    plt.show()
    plt.close()


In [ ]:
df_hpccg

In [ ]:
# Read the file while skipping unwanted lines
with open(DATAPATH + "/results_tealeaf.csv") as f:
    # tealeaf tries to verify the solution but fails to read the problems file in my experiment
    lines = [line.strip() for line in f if not line.startswith("Command exited")]

# Now read the cleaned lines into a DataFrame
from io import StringIO

cleaned_data = "\n".join(lines)
df_tealeaf = pd.read_csv(StringIO(cleaned_data), header=None, names=["size", "iterations", "threads", "mode", "time"])

# Optional: convert time column from "Xs" to float
df_tealeaf["time"] = df_tealeaf["time"].str.rstrip("s").astype(float)




In [ ]:
df_tealeaf_filtered = df_tealeaf[df_tealeaf["iterations"] == 2]

In [ ]:
# Create the line plot
sns.lineplot(data=df_tealeaf_filtered, x="size", y="time", hue="mode", marker="o")

plt.title(f"Tealeaf: Execution Time")
plt.xlabel("Size")
plt.ylabel("Time (s)")
plt.legend(title="Tsan build")
plt.tight_layout()
plt.savefig("tealeaf_time.pdf")

In [ ]:
for size_val, df_group in df_tealeaf_filtered.groupby("size"):
    sns.lineplot(data=df_group, x="threads", y="time", hue="mode", marker="o")

    plt.title(f"Tealeaf: Execution Time (size={size_val})")
    plt.xlabel("num_threads")
    plt.ylabel("Time (s)")
    plt.legend(title="Tsan build")
    plt.tight_layout()
    plt.savefig(f"tealeaf_time_size{size_val}.pdf")
    plt.show()
    plt.close()

In [ ]:
mean_times = df_tealeaf_filtered.groupby(["size", "mode"])["time"].mean().reset_index()
# Pivot to wide format*: one row per config, columns per mode
pivoted = mean_times.pivot(index="size", columns="mode", values="time")
# Normalize all modes relative to 'without'
percentages = pivoted.div(pivoted["without"], axis=0)

percentages[["modified", "normal"]].plot(kind="line", marker="o")

plt.title(f"TeaLeaf: Slowdown")
plt.xlabel("problem Size")
plt.ylabel("Slowdown Factor")
plt.legend(title="Tsan build")
plt.tight_layout()
plt.savefig("tealeaf_slowdown.pdf")

In [ ]:
df_fildered = df_tealeaf_filtered[df_tealeaf_filtered["size"] < 1024]  # timeout on larger measurement
for size_val, df_group in df_tealeaf_filtered.groupby("size"):
    mean_times = df_group.groupby(["threads", "mode"])["time"].mean().reset_index()
    # Pivot to wide format*: one row per config, columns per mode
    pivoted = mean_times.pivot(index="threads", columns="mode", values="time")
    # Normalize all modes relative to 'without'
    percentages = pivoted.div(pivoted["without"], axis=0)

    percentages[["modified", "normal"]].plot(kind="line", marker="o")

    plt.title(f"TeaLeaf: Slowdown (size={size_val})")
    plt.xlabel("num_threads")
    plt.ylabel("Slowdown Factor")
    plt.legend(title="Tsan build")
    plt.tight_layout()
    plt.savefig(f"tealeaf_slowdown_size_{size_val}.pdf")
    plt.show()
    plt.close()